# Structured Extraction Experiments

**Goal:** Validate that the chosen LLM (`llama-3.3-70b-versatile` via Groq) can reliably conform to the Pydantic `ResumeExtractionResult` schema using the text produced by the proven PyMuPDF / python-docx parsers.

This is a **contract validation step**. We need to confirm the LLM can actually generate that structure consistently.

### What We Are Testing
1. **Schema Definition** — Formally define the `ResumeExtractionResult` Pydantic model that Node 2 (Extractor) of the LangGraph pipeline will produce.
2. **LLM Call** — Use `ChatGroq` + `.with_structured_output()` to force the Groq API to return data matching that schema.
3. **The Test** — Feed it the perfectly parsed text from sample resumes and inspect the output.

### Why This Matters
The schema define here becomes the **typed contract** that flows through the entire LangGraph state. If the LLM cannot reliably produce it, we need to know now — not after we've built the pipeline around it.

---
## 1. Setup & Imports

In [1]:
import os
import fitz  # PyMuPDF
import docx
from pathlib import Path
from dotenv import load_dotenv

# LangChain / Groq
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate

# Pydantic
from pydantic import BaseModel, Field
from typing import Optional

# --- Load environment variables from .env ---
load_dotenv()
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

if not GROQ_API_KEY:
    raise EnvironmentError("GROQ_API_KEY not found. Make sure .env is set up correctly.")
print("✅ GROQ_API_KEY loaded successfully.")

# --- Set up paths relative to the notebook ---
BASE_DIR = Path().resolve().parent
RESUMES_DIR = BASE_DIR / "data" / "raw" / "resumes"
print(f"📁 Resumes directory: {RESUMES_DIR}")

✅ GROQ_API_KEY loaded successfully.
📁 Resumes directory: /Users/neehanth/Documents/Data Scientist/agentic_resume_screener_project/agentic_resume_screener/data/raw/resumes


---
## 2. Define the Pydantic Schema (`ResumeExtractionResult`)

This is the **typed contract** for LangGraph Node 2 (the Extractor).

> **Design decision:** We use nested Pydantic models (`ExperienceEntry`, `EducationEntry`) rather than plain strings. This makes downstream processing in the Analyzer and Recommender nodes significantly easier — each field is directly addressable without additional parsing.

In [2]:
class ExperienceEntry(BaseModel):
    """A single role in the candidate's professional history."""
    job_title: str = Field(description="The candidate's job title for this role.")
    company: str = Field(description="The name of the employer or organization.")
    duration: Optional[str] = Field(
        default=None,
        description="The employment period, e.g. '2022–Present' or '2019–2022'."
    )
    responsibilities: list[str] = Field(
        description="List of key responsibilities and accomplishments in this role, as bullet point strings."
    )


class EducationEntry(BaseModel):
    """A single education credential."""
    degree: str = Field(description="The degree or certification earned, e.g. 'M.S., Computer Science'.")
    institution: str = Field(description="The name of the university or institution.")
    graduation_year: Optional[str] = Field(default=None, description="The year of graduation, e.g. '2020'.")


class ResumeExtractionResult(BaseModel):
    """
    The fully structured output of the Extractor Node (Node 2).
    Represents all key information extracted from a single candidate's resume.
    """
    candidate_name: str = Field(description="The full name of the candidate.")
    contact_email: Optional[str] = Field(default=None, description="The candidate's email address.")
    contact_phone: Optional[str] = Field(default=None, description="The candidate's phone number as it appears on the resume, e.g. '(267) 555-0142'.")
    professional_summary: str = Field(
        description="A concise summary of the candidate's professional background, written in their own words or closely paraphrased. Max 3–4 sentences."
    )
    total_years_of_experience: Optional[float] = Field(
        default=None,
        description="Estimated total years of relevant professional experience, calculated from the resume dates. Use a float, e.g. 5.0 or 2.5."
    )
    technical_skills: list[str] = Field(
        description="A flat list of all specific technical skills, tools, libraries, frameworks, and programming languages mentioned anywhere in the resume."
    )
    soft_skills: list[str] = Field(
        default_factory=list,
        description="A list of soft skills or professional competencies inferred from the resume, e.g. 'stakeholder communication', 'mentoring'."
    )
    experience: list[ExperienceEntry] = Field(
        description="A list of all professional roles, ordered from most recent to oldest."
    )
    education: list[EducationEntry] = Field(
        description="A list of all educational credentials, ordered from highest to lowest degree."
    )
    certifications: list[str] = Field(
        default_factory=list,
        description="A list of any professional certifications mentioned, e.g. 'AWS Certified Solutions Architect'."
    )
    notable_projects: list[str] = Field(
        default_factory=list,
        description="High-level names or one-sentence descriptions of notable personal or professional projects."
    )


print("✅ ResumeExtractionResult schema defined.")
print(f"   Fields: {list(ResumeExtractionResult.model_fields.keys())}")

✅ ResumeExtractionResult schema defined.
   Fields: ['candidate_name', 'contact_email', 'contact_phone', 'professional_summary', 'total_years_of_experience', 'technical_skills', 'soft_skills', 'experience', 'education', 'certifications', 'notable_projects']


---
## 3. Reuse the Proven Parsers from Notebook 01

These functions are the **final, validated** parsers from the document parsing experiments. Reproduced here for self-containment; they will eventually live in `src/core/parser.py`.

> **Note on `parse_docx`:** This uses the XML traversal approach (`iter_block_items`) — **not** the naive two-pass version. It walks the raw Word XML body in true sequential order so that tables are interleaved with paragraphs exactly as they appear visually. This is critical for resumes like Maya's, where the `CORE SKILLS` table sits between `PROFESSIONAL SUMMARY` and `EDUCATION` — not at the bottom of the document.

In [3]:
# --- Helper: Deterministic experience calculator ---
# Future home: src/utils/experience.py
import re
from datetime import datetime
from dateutil.relativedelta import relativedelta


def _parse_date(s: str) -> datetime:
    s = s.strip()
    if s.lower() in ("present", "now", "current"):
        return datetime.now()
    for fmt in ("%b %Y", "%B %Y", "%Y-%m", "%Y/%m"):
        try:
            return datetime.strptime(s, fmt)
        except ValueError:
            continue
    if re.fullmatch(r"\d{4}", s):
        return datetime(int(s), 1, 1)
    raise ValueError(f"Unrecognized date format: {s!r}")


def calculate_experience_years(experience: list) -> float:
    """
    Calculate total professional experience from a list of ExperienceEntry objects.

    Returns a float encoded as {years}.{months} where the decimal part
    is the raw remaining month count — NOT a mathematical fraction:
        5 yrs 0 mo  → 5.0
        5 yrs 6 mo  → 5.6
        3 yrs 11 mo → 3.11
    """
    total_months = 0

    for entry in experience:
        if not entry.duration:
            continue

        raw = entry.duration.replace("–", "-").replace("—", "-")
        parts = re.split(r"\s*-\s*", raw, maxsplit=1)

        if len(parts) != 2:
            print(f"  ⚠️  Skipping unparseable duration: {entry.duration!r}")
            continue

        try:
            start_dt = _parse_date(parts[0])
            end_dt   = _parse_date(parts[1])
        except ValueError as e:
            print(f"  ⚠️  {e} — skipping: {entry.job_title!r}")
            continue

        delta = relativedelta(end_dt, start_dt)
        role_months = delta.years * 12 + delta.months

        if role_months > 0:
            total_months += role_months
            print(f"  📌 {entry.job_title} @ {entry.company} ({entry.duration}) → {role_months} mo")

    years = total_months // 12
    rem_months = total_months % 12
    result = float(f"{years}.{rem_months}")

    print(f"\n  ✅ Total: {total_months} mo → {years} yrs {rem_months} mo → {result}")
    return result

print("✅ calculate_experience_years() loaded.")


✅ calculate_experience_years() loaded.


In [4]:
from docx.oxml.text.paragraph import CT_P
from docx.oxml.table import CT_Tbl
from docx.text.paragraph import Paragraph
from docx.table import Table

def parse_pdf(file_path: Path) -> str:
    """
    Extract text from a PDF using PyMuPDF block-level extraction.
    Preserves column order for multi-column layouts.
    """
    text = ""
    try:
        doc = fitz.open(file_path)
        for page in doc:
            blocks = page.get_text("blocks")
            for block in blocks:
                text += block[4]
        return text.strip()
    except Exception as e:
        print(f"Error reading {file_path}: {e}")
        return ""


def iter_block_items(parent):
    """
    Yields each paragraph and table child within *parent*, in document order.
    Solves the python-docx hierarchy issue by reading the raw XML sequence
    instead of iterating doc.paragraphs and doc.tables as two separate lists.
    """
    if isinstance(parent, docx.document.Document):
        parent_elm = parent.element.body
    elif isinstance(parent, docx.table._Cell):
        parent_elm = parent._tc
    else:
        raise ValueError("Unsupported parent type")

    for child in parent_elm.iterchildren():
        if isinstance(child, CT_P):
            yield Paragraph(child, parent)
        elif isinstance(child, CT_Tbl):
            yield Table(child, parent)


def parse_docx(file_path: Path) -> str:
    """
    Extract text from a DOCX file using XML traversal to preserve document hierarchy.
    Walks doc.element.body children in order, interleaving paragraphs and tables
    exactly as they appear in the document — NOT paragraphs-first, tables-second.
    """
    try:
        doc = docx.Document(file_path)
        full_text = []

        for block in iter_block_items(doc):
            if isinstance(block, Paragraph):
                if block.text.strip():
                    full_text.append(block.text.strip())

            elif isinstance(block, Table):
                for row in block.rows:
                    row_data = []
                    for cell in row.cells:
                        clean_cell = cell.text.strip()
                        if clean_cell and clean_cell not in row_data:
                            row_data.append(clean_cell)
                    if row_data:
                        full_text.append("\n".join(row_data))

        return "\n".join(full_text)

    except Exception as e:
        print(f"Error reading {file_path}: {e}")
        return ""


def parse_resume(file_path: Path) -> str:
    """Route a resume file to the correct parser based on extension."""
    suffix = file_path.suffix.lower()
    if suffix == ".pdf":
        return parse_pdf(file_path)
    elif suffix == ".docx":
        return parse_docx(file_path)
    else:
        raise ValueError(f"Unsupported file type: {suffix}")


print("✅ Parsers loaded (DOCX using XML traversal — hierarchy preserved).")

✅ Parsers loaded (DOCX using XML traversal — hierarchy preserved).


---
## 4. Build the Extraction Chain

We use `ChatGroq` with `.with_structured_output()` — the standard LangChain pattern for enforcing Pydantic schemas.

> **Why this approach is correct for LangGraph:** In the final pipeline, each LangGraph node will receive a `ChatGroq` instance and call `.with_structured_output(SomeSchema)` exactly like this. This experiment is a 1:1 preview of Node 2's actual implementation — not a prototype we'll have to rewrite.

In [5]:
# --- Instantiate the LLM ---
# temperature=0 for deterministic, structured output.
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0,
    api_key=GROQ_API_KEY
)

# --- Patch the LLM to enforce the Pydantic schema ---
structured_llm = llm.with_structured_output(ResumeExtractionResult)

# --- Define the system prompt ---
# This will become the versioned prompt file: prompts/v1_resume_extractor.txt
SYSTEM_PROMPT = """\
You are an expert resume parser and information extractor. Your task is to extract 
structured information from the resume text provided by the user.
Rules:
- Extract information ONLY from the provided resume text. Do not infer or invent details.
- For 'total_years_of_experience', calculate it from the date ranges in the Experience section. 
  If dates are ambiguous, make a conservative best estimate and use a float (e.g., 5.0).
- For 'technical_skills', extract every specific tool, library, language, framework, or platform 
  mentioned anywhere in the document — including project descriptions and structured sidebar 
  sections. Do NOT omit any tool from a structured list. Prefer concrete tool names over 
  methodology labels.
- For 'professional_summary', use the candidate's own summary if present. If not, synthesize 
  one from their experience. Keep it to 3-4 sentences max.
- For 'soft_skills', do NOT require explicit mentions. Infer them from accomplishment language and contextual evidence in the resume. Extract only specific  competencies. Avoid generic filler. Prefer precise, evidence-backed phrasing.
- If a field is not found in the resume, use an empty list [] or null as appropriate.
"""

# --- Build the prompt template ---
prompt_template = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human", "Please extract the structured information from the following resume:\n\n{resume_text}")
])

# --- Compose the full extraction chain ---
extraction_chain = prompt_template | structured_llm

print("✅ Extraction chain built: prompt_template | structured_llm")

✅ Extraction chain built: prompt_template | structured_llm


---
## 5. Test 1 — DOCX Resume: `maya_srinivasan` (Senior AI/LLM Engineer)

Maya is one of the strongest candidate — a PhD with 5+ years of direct LLM experience. The LLM should extract a rich, well-populated schema.

In [6]:
import time

maya_path = RESUMES_DIR / "maya_srinivasan_res_aiml_c01.docx"
maya_text = parse_resume(maya_path)

print(f"📄 Parsed resume: {maya_path.name}")
print(f"   Character count: {len(maya_text):,}")
print(f"   Preview (first 300 chars):\n{'-'*50}\n{maya_text[:300]}\n")

📄 Parsed resume: maya_srinivasan_res_aiml_c01.docx
   Character count: 5,162
   Preview (first 300 chars):
--------------------------------------------------
Dr. Maya Srinivasan
Philadelphia, PA  |  maya.srinivasan.ai@gmail.com  |  (267) 555-0142  |  github.com/mayasri-ai  |  mayasrinivasan.dev  |  linkedin.com/in/mayasrinivasan
PROFESSIONAL SUMMARY
AI/LLM Engineer and applied NLP researcher with 5+ years of experience building, fine-tuning, evaluating, 



In [7]:
import json

print("🚀 Calling Groq API with structured output...")
start = time.time()

maya_result: ResumeExtractionResult = extraction_chain.invoke({"resume_text": maya_text})

latency = time.time() - start
print(f"✅ Extraction complete. Latency: {latency:.2f}s\n")

# Pretty-print the Pydantic model as JSON
print(json.dumps(maya_result.model_dump(), indent=2))

🚀 Calling Groq API with structured output...
✅ Extraction complete. Latency: 1.97s

{
  "candidate_name": "Dr. Maya Srinivasan",
  "contact_email": "maya.srinivasan.ai@gmail.com",
  "contact_phone": "(267) 555-0142",
  "professional_summary": "AI/LLM Engineer and applied NLP researcher with 5+ years of experience building, fine-tuning, evaluating, and deploying large language model systems for enterprise use cases. Strong background in retrieval-augmented generation (RAG), prompt engineering, model optimization, agentic workflows, and scalable cloud deployment. Experienced partnering with cross-functional stakeholders to translate ambiguous business challenges into production-ready AI systems.",
  "total_years_of_experience": 5.0,
  "technical_skills": [
    "Hugging Face",
    "LangChain",
    "LangGraph",
    "OpenAI API",
    "Claude API",
    "LLaMA",
    "vLLM",
    "PyTorch",
    "TensorFlow",
    "PEFT",
    "LoRA",
    "DPO",
    "supervised fine-tuning",
    "embedding models"

In [ ]:
print("--- Maya Srinivasan ---")
calculated = calculate_experience_years(maya_result.experience)
print(f"  Experience Calculated: {calculated}")

maya_result.total_years_of_experience = calculated
maya_result.soft_skills = [s.lower() for s in maya_result.soft_skills]

print(json.dumps(maya_result.model_dump(), indent=2))

--- Maya Srinivasan ---
  📌 Senior AI/LLM Engineer @ Stratagem Analytics (2022-Present) → 50 mo
  📌 Applied NLP Engineer @ Vantage Decision Systems (2020-2022) → 24 mo

  ✅ Total: 74 mo → 6 yrs 2 mo → 6.2
  Experience Calculated: 6.2
{
  "candidate_name": "Dr. Maya Srinivasan",
  "contact_email": "maya.srinivasan.ai@gmail.com",
  "contact_phone": "(267) 555-0142",
  "professional_summary": "AI/LLM Engineer and applied NLP researcher with 5+ years of experience building, fine-tuning, evaluating, and deploying large language model systems for enterprise use cases. Strong background in retrieval-augmented generation (RAG), prompt engineering, model optimization, agentic workflows, and scalable cloud deployment. Experienced partnering with cross-functional stakeholders to translate ambiguous business challenges into production-ready AI systems.",
  "total_years_of_experience": 6.2,
  "technical_skills": [
    "Hugging Face",
    "LangChain",
    "LangGraph",
    "OpenAI API",
    "Claude A

`Insights:` Test 1 — Maya Srinivasan (DOCX)

- **✅ Schema valid.** All fields populated correctly. No hallucinations detected on manual read.
- **Technical skills (43):** XML traversal paid off — skills from the CORE SKILLS table were extracted in place, not dumped at the end. Project and experience bullets contributed additional tools beyond the explicit skills section.
- **Soft skills (4):** `stakeholder communication`, `mentoring`, `experimentation discipline`, and `technical writing` — all correctly inferred from accomplishment language. Clean, evidence-backed competencies with no generic filler.
- **`total_years_of_experience` — fixed:** LLM outputs `5.0` (anchored to *"5+ years"* in the summary). `calculate_experience_years()` correctly computed `74 months → 6 yrs 2 mo → 6.2`, which overwrites the schema value. Deterministic math > LLM self-report.
- **Latency: 2.73s** — within target. Slight increase from prior run (~1.67s) due to LLM non-determinism; both are well within the <15s budget.

---
## 6. Test 2 — Two-Column PDF Resume: `jordan_kim` (Junior AI Engineer)

Jordan's resume uses a two-column layout — the same one that broke `pdfplumber` in Notebook 01. We're verifying that our PyMuPDF parser produces text clean enough for the LLM to extract structured data from, despite the complex layout. This is an **edge case test**.

In [9]:
jordan_path = RESUMES_DIR / "jordan_kim_res_aiml_c02.pdf"
jordan_text = parse_resume(jordan_path)

print(f"📄 Parsed resume: {jordan_path.name}")
print(f"   Character count: {len(jordan_text):,}")
print(f"   Preview:\n{jordan_text}")

📄 Parsed resume: jordan_kim_res_aiml_c02.pdf
   Character count: 3,662
   Preview:
Jordan Kim
Junior AI Engineer  •  LLM Prototyping  •  Applied NLP
Philadelphia, PA
jordan.kim.ml@gmail.com
(484) 555-0198
github.com/jordankim-ai
jordankim.dev | 
linkedin.com/in/jordankim
FIT SNAPSHOT
• Strong technical overlap 
with the role
• Hands-on with RAG, prompt 
engineering, and APIs
• Gap: does not yet meet 2+ 
years post-master's 
experience
CORE TOOLS
LLM: OpenAI, Claude, LLaMA
Frameworks: Hugging Face, 
LangChain
ML: PyTorch, TensorFlow
Infra: AWS, Docker, FastAPI
Search: FAISS, ChromaDB
Code: Python, SQL, Git
COMMUNITY
• University research 
publication (2025)
• Open-source LangChain 
contribution
• Technical blogging on RAG 
and evaluation
Professional Summary
Junior AI Engineer with a recent Master’s in Computer Science and hands-on 
experience building LLM-based applications, RAG pipelines, and prompt-driven 
tools in academic and internship settings. Strong Python and PyTorch backgroun

In [10]:
print("🚀 Calling Groq API with structured output...")
start = time.time()

jordan_result: ResumeExtractionResult = extraction_chain.invoke({"resume_text": jordan_text})

latency = time.time() - start
print(f"✅ Extraction complete. Latency: {latency:.2f}s\n")

print(json.dumps(jordan_result.model_dump(), indent=2))

🚀 Calling Groq API with structured output...
✅ Extraction complete. Latency: 1.31s

{
  "candidate_name": "Jordan Kim",
  "contact_email": "jordan.kim.ml@gmail.com",
  "contact_phone": "(484) 555-0198",
  "professional_summary": "Junior AI Engineer with a recent Master\u2019s in Computer Science and hands-on experience building LLM-based applications, RAG pipelines, and prompt-driven tools in academic and internship settings. Strong Python and PyTorch background with practical exposure to Hugging Face, LangChain, OpenAI APIs, and cloud deployment. Comfortable working in fast-moving, research-oriented environments and translating business problems into working prototypes.",
  "total_years_of_experience": 1.5,
  "technical_skills": [
    "Python",
    "PyTorch",
    "Hugging Face",
    "LangChain",
    "OpenAI",
    "FAISS",
    "Docker",
    "FastAPI",
    "SQL",
    "Git",
    "AWS",
    "ChromaDB"
  ],
  "soft_skills": [
    "Collaboration",
    "Communication",
    "Problem-solving",

In [ ]:
print("--- Jordan Kim ---")
calculated = calculate_experience_years(jordan_result.experience)
print(f"\n  LLM reported : {jordan_result.total_years_of_experience}")
print(f"  Calculated   : {calculated}")

jordan_result.total_years_of_experience = calculated
jordan_result.soft_skills = [s.lower() for s in jordan_result.soft_skills]

print(json.dumps(jordan_result.model_dump(), indent=2))

--- Jordan Kim ---
  📌 AI Engineer Intern @ Novus Labs (2025–Present) → 14 mo
  📌 Graduate Research Assistant @ Northeastern University (2024–2025) → 12 mo

  ✅ Total: 26 mo → 2 yrs 2 mo → 2.2

  LLM reported : 2.2
  Calculated   : 2.2
{
  "candidate_name": "Jordan Kim",
  "contact_email": "jordan.kim.ml@gmail.com",
  "contact_phone": "(484) 555-0198",
  "professional_summary": "Junior AI Engineer with a recent Master\u2019s in Computer Science and hands-on experience building LLM-based applications, RAG pipelines, and prompt-driven tools in academic and internship settings. Strong Python and PyTorch background with practical exposure to Hugging Face, LangChain, OpenAI APIs, and cloud deployment. Comfortable working in fast-moving, research-oriented environments and translating business problems into working prototypes.",
  "total_years_of_experience": 2.2,
  "technical_skills": [
    "Python",
    "PyTorch",
    "Hugging Face",
    "LangChain",
    "OpenAI",
    "FAISS",
    "Docker",

`Insights:` Test 2 — Jordan Kim (Two-Column PDF)

- **✅ Schema valid.** All fields populated including `contact_phone: "(484) 555-0198"` — correctly lifted from the two-column PDF header.

- **PDF parser held up.** PyMuPDF's block extraction produced clean text despite the two-column layout. The `FIT SNAPSHOT` sidebar was correctly ignored — not treated as real experience.

- **CORE TOOLS sidebar:** All 12 concrete tools extracted correctly (`AWS`, `FAISS`, `ChromaDB`, `Docker`, `FastAPI`, etc.). Previous run had `AWS` silently dropped and replaced by methodology labels (`RAG`, `prompt engineering`) — fixed by tightening the `technical_skills` prompt rule.

- **Soft skills (4):** `collaboration`, `communication`, `problem-solving`, `adaptability` — declared on the resume (junior candidate pattern). Normalized to lowercase via post-processing.

- **`total_years_of_experience` — fixed:** LLM varies between `1.0–1.5` (non-deterministic, anchored to internship). `calculate_experience_years()` deterministically computed `26 months → 2 yrs 2 mo → 2.2` by summing both roles.

- **Latency: 1.31s** — fastest run so far. Both candidates well within the <15s target.

---
## 7. Schema Validation Check

Since `.with_structured_output()` raises a validation error if the schema cannot be met, any successful call above already proves the contract. This cell does an explicit programmatic check to confirm all non-optional fields are populated.

In [ ]:
def validate_extraction(label: str, result: ResumeExtractionResult) -> None:
    """Print a structured validation report for an extraction result."""
    print(f"\n{'='*50}")
    print(f"Validation Report: {label}")
    print(f"{'='*50}")
    
    checks = {
        "candidate_name populated": bool(result.candidate_name),
        "professional_summary populated": bool(result.professional_summary),
        "technical_skills populated": len(result.technical_skills) > 0,
        "experience list non-empty": len(result.experience) > 0,
        "education list non-empty": len(result.education) > 0,
        "all experience entries have responsibilities": all(
            len(e.responsibilities) > 0 for e in result.experience
        ),
    }
    
    all_passed = True
    for check, passed in checks.items():
        status = "✅" if passed else "❌"
        print(f"  {status} {check}")
        if not passed:
            all_passed = False
    
    print(f"\n  📊 Technical skills extracted: {len(result.technical_skills)}")
    print(f"  📊 Roles extracted: {len(result.experience)}")
    print(f"  📊 Education entries: {len(result.education)}")
    print(f"  📊 Projects: {len(result.notable_projects)}")
    print(f"\n  {'🎉 All checks passed!' if all_passed else '⚠️  Some checks failed — review output.'}")


validate_extraction("Maya Srinivasan (DOCX)", maya_result)
validate_extraction("Jordan Kim (Two-Column PDF)", jordan_result)

---
## 8. Summary & Next Steps

### Contract Validation Result
*(Fill in after running)*

| Resume | File Type | Schema Valid? | Latency | Notes |
|--------|-----------|:-------------:|---------|-------|
| Maya Srinivasan | DOCX | ✅ / ❌ | ~Xs | |
| Jordan Kim | Two-Column PDF | ✅ / ❌ | ~Xs | |

### Decisions Made
- **Schema is validated** — `ResumeExtractionResult` is ready to be formalized in `src/schemas/resume.py`.
- **Prompt v1** is captured in the `SYSTEM_PROMPT` constant above — ready to be moved to `prompts/v1_resume_extractor.txt`.

### Next Steps
- [ ] **Phase 4 remaining:** Create the golden evaluation dataset (20 JD/resume pairs) using this schema as the labeling structure.
- [ ] **Phase 5:** Write `02_jd_extraction.ipynb` to define and validate the `JDExtractionResult` schema for the Job Description side.
- [ ] **Phase 7:** Move `ResumeExtractionResult` into `src/schemas/resume.py` and the parsers into `src/core/parser.py`.